# Lab 4: Backpropagation (2-3-1 Neural Network)

This notebook is designed to be reusable.

- Default dataset loads from `data/xor.csv`.
- To use your own dataset, set `DATA_PATH` to your CSV.
- Weights are initialized from `SEED` (set `SEED = None` for new random init each run).


In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

# ---- Config (edit these) ----
DATA_PATH = Path("data/xor.csv")
TARGET_COL = "y"
SEED = 42  # set to None for non-deterministic init
HIDDEN_UNITS = 3
LEARNING_RATE = 0.5
EPOCHS = 100_000
PRINT_WEIGHTS = True

# ---- Load dataset ----
if DATA_PATH.exists():
    df = pd.read_csv(DATA_PATH)
    if TARGET_COL not in df.columns:
        raise ValueError(f"TARGET_COL '{TARGET_COL}' not found in {DATA_PATH}")
    X = df.drop(columns=[TARGET_COL]).to_numpy(dtype=float)
    y = df[TARGET_COL].to_numpy(dtype=float).reshape(-1, 1)
else:
    # Fallback XOR dataset
    X = np.array([[0, 0], [0, 1], [1, 0], [1, 1]], dtype=float)
    y = np.array([[0], [1], [1], [0]], dtype=float)
    print(f"Note: {DATA_PATH} not found; using built-in XOR dataset instead.")

# Sigmoid activation function
def sigmoid(x):
    return 1 / (1 + np.exp(-x))

# Derivative of sigmoid (expects sigmoid output)
def sigmoid_derivative(sigmoid_output):
    return sigmoid_output * (1 - sigmoid_output)


In [ ]:
# Initialize weights and biases
rng = np.random.default_rng(SEED)
n_inputs = X.shape[1]

input_weights = rng.uniform(size=(n_inputs, HIDDEN_UNITS))
hidden_weights = rng.uniform(size=(HIDDEN_UNITS, 1))
hidden_bias = rng.uniform(size=(1, HIDDEN_UNITS))
output_bias = rng.uniform(size=(1, 1))
lr = LEARNING_RATE

if PRINT_WEIGHTS:
    print("Initial Input to Hidden Weights:")
    print(input_weights)
    print()
    print("Initial Hidden to Output Weights:")
    print(hidden_weights)
    print()
    print("Initial Hidden Layer Biases:")
    print(hidden_bias)
    print()
    print("Initial Output Layer Bias:")
    print(output_bias)


In [ ]:
# Train the neural network
for epoch in range(EPOCHS):
    # Forward pass
    hidden_layer_input = np.dot(X, input_weights) + hidden_bias
    hidden_layer_output = sigmoid(hidden_layer_input)
    output_layer_input = np.dot(hidden_layer_output, hidden_weights) + output_bias
    output = sigmoid(output_layer_input)

    # Error
    error = y - output

    # Backpropagation
    d_output = error * sigmoid_derivative(output)
    error_hidden_layer = d_output.dot(hidden_weights.T)
    d_hidden_layer = error_hidden_layer * sigmoid_derivative(hidden_layer_output)

    # Update weights and biases
    hidden_weights += hidden_layer_output.T.dot(d_output) * lr
    output_bias += np.sum(d_output, axis=0, keepdims=True) * lr
    input_weights += X.T.dot(d_hidden_layer) * lr
    hidden_bias += np.sum(d_hidden_layer, axis=0, keepdims=True) * lr

print("Final Output after Training:")
print(output)

if PRINT_WEIGHTS:
    print()
    print("Final Input to Hidden Weights:")
    print(input_weights)
    print()
    print("Final Hidden to Output Weights:")
    print(hidden_weights)
    print()
    print("Final Hidden Layer Biases:")
    print(hidden_bias)
    print()
    print("Final Output Layer Bias:")
    print(output_bias)


In [ ]:
# Test a single input
test_input = np.array([0, 0], dtype=float)

hidden_layer_input = np.dot(test_input, input_weights) + hidden_bias
hidden_layer_output = sigmoid(hidden_layer_input)
output_layer_input = np.dot(hidden_layer_output, hidden_weights) + output_bias
test_output = sigmoid(output_layer_input)

print("Test Input:", test_input.tolist())
print("Output after training:", test_output)
